In [1]:

import asyncio
import os
import sys
import random
from typing import Union, List, Optional, Type

import pandas as pd
from openai import AsyncOpenAI
from pydantic import BaseModel
from dotenv import load_dotenv

In [2]:
n_questions_per_chunk: int = 3
max_chunks: int = 5

In [3]:
from config import VECTORSTORE_DIR, MODEL_NAME, TOP_K
from pipeline import load_vectorstore

load_dotenv()
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")
client = AsyncOpenAI(api_key=os.getenv("OPENAI_API_KEY"))

# output
EVAL_OUTPUT_CSV = os.path.join(
    os.getcwd(),
    "qa_eval_results.csv"
)

### UTILITAIRE LLM

In [4]:
async def run_llm_call(
    system_prompt: str,
    user_prompt: str,
    response_model: Optional[Type[BaseModel]] = None,
    model: str = MODEL_NAME,
) -> Union[str, BaseModel, None]:
    max_retries = 5
    base_delay  = 10

    for attempt in range(0, max_retries):
        try:
            if response_model:
                response = await client.beta.chat.completions.parse(
                    model=model,
                    messages=[
                        {"role": "system", "content": system_prompt.strip()},
                        {"role": "user",   "content": user_prompt.strip()},
                    ],
                    temperature=0,
                    response_format=response_model,
                )
                return response.choices[0].message.parsed
            else:
                response = await client.chat.completions.create(
                    model=model,
                    messages=[
                        {"role": "system", "content": system_prompt.strip()},
                        {"role": "user",   "content": user_prompt.strip()},
                    ],
                    temperature=0,
                )
                return response.choices[0].message.content

        except Exception as e:
            print(f"  [LLM attempt {attempt}/{max_retries}] Error: {e}")
            if attempt == max_retries:
                return None
            await asyncio.sleep(300)


async def run_llm_calls(
    system_prompts: List[str],
    user_prompts: List[str],
    response_model: Optional[Type[BaseModel]] = None,
    model: str = MODEL_NAME,
) -> List:
    """parallel LLM calls """
    tasks = [
        run_llm_call(sp, up, response_model=response_model, model=model)
        for sp, up in zip(system_prompts, user_prompts)
    ]
    return await asyncio.gather(*tasks)

### 1. extract chunks + generate Q&A

In [5]:
class QAPair(BaseModel):
    question:   str
    answer:     str   # ground truth
    chunk_used: str   # chunk source


class QAList(BaseModel):
    pairs: List[QAPair]

In [6]:
SYSTEM_QA_GENERATION = """
You are an expert at creating evaluation datasets from scientific article excerpts.

Given a passage from a scientific article, generate {n_questions} question/answer pairs

Rules:
- Questions must be precise and answerable from the passage alone
- Answers must be factually grounded in the passage (no inference beyond the text)
- Vary question types: factual, comparative, causal, definitional
- Return valid JSON only
"""

USER_QA_GENERATION = """
Article passage:

{chunk}

Generate {n_questions} question/answer pairs from this passage.
"""

In [7]:
def get_all_chunks_from_faiss() -> List[str]:
    """
    load all text chunks stored in the FAISS vector store
    """
    vectorstore = load_vectorstore()
    docs   = list(vectorstore.docstore._dict.values())
    chunks = [doc.page_content for doc in docs if doc.page_content.strip()]
    print(f"{len(chunks)} chunks extracted from faiss")
    return chunks

In [8]:
async def generate_qa_from_vectorstore(
    n_questions_per_chunk: int = 3,
    max_chunks: int = 5,
) -> pd.DataFrame:
    """
    generate Q&A dataset from FAISS chunks

    Args:
        n_questions_per_chunk: number of Q&A generated per chunk
        max_chunks: limit to prevent too many LLM calls

    Returns:
        df with values for question, answer_ref, context
    """
    chunks = get_all_chunks_from_faiss()[:max_chunks]
    sys_prompts  = [SYSTEM_QA_GENERATION.format(n_questions=n_questions_per_chunk)] * len(chunks)
    user_prompts = [
        USER_QA_GENERATION.format(chunk=chunk, n_questions=n_questions_per_chunk)
        for chunk in chunks
    ]

    responses = await run_llm_calls(sys_prompts, user_prompts, response_model=QAList)

    rows = []
    for resp in responses:
        if resp:
            for pair in resp.pairs:
                rows.append({
                    "question":   pair.question,
                    "answer_ref": pair.answer,
                    "context":    pair.chunk_used,
                })

    df = pd.DataFrame(rows)
    print(f"{len(df)} Q&A generated pairs")
    return df

In [9]:
df_qsts = await generate_qa_from_vectorstore(
        n_questions_per_chunk=n_questions_per_chunk,
        max_chunks=max_chunks,
)

78 chunks extracted from faiss
15 Q&A generated pairs


In [10]:
df_qsts

,question,answer_ref,context
0,What is the main purpose of the AI surrogate i...,To simulate coastal tidal wave propagation in ...,A Fast AI Surrogate for Coastal Ocean Circulat...
1,What are the limitations of traditional coasta...,"They are time-consuming and energy expensive, ...","However, the process is time-consuming and ene..."
2,What technology does the AI surrogate utilize ...,It utilizes a 4D Swin Transformer and incorpor...,This paper introduces an AI surrogate based on...
3,What technology does the AI surrogate use to s...,A 4D Swin Transformer,This paper introduces an AI surrogate based on...
4,How much time does the AI surrogate reduce the...,"From 9,908 seconds to 22 seconds, achieving ov...",Our experiments demonstrate that our AI surrog...
5,What is the primary purpose of simulating ocea...,To develop early warning systems that save liv...,Oceanographers simulate ocean current circulat...
6,What is the main focus of the article regardin...,The main focus is on AI (neural network) surro...,The emergence of Graphics Processing Units (GP...
7,How long does it take to simulate 12 days’ for...,"It takes around 9,908 seconds to simulate 12 d...",Our preliminary results show that with 512 CPU...
8,What are the challenges mentioned in applying ...,"Several unique challenges exist, including the...","Unfortunately, these surrogate models cannot b..."
9,What is the primary focus of the AI surrogate ...,The primary focus of the AI surrogate is to mo...,"For example, vision transformers have been tra..."


### 2- RAG answers

In [11]:
async def generate_rag_answers(df: pd.DataFrame) -> pd.DataFrame:
    """
    call oceanAI rag to answer the generated questions 
    Returns:
        previous dataframe + context_retrieved, answer_rag
    """
    from main import build_rag_chain
 
    vectorstore = load_vectorstore()
    chain       = build_rag_chain(vectorstore)
    answers  = []
    contexts = []
 
    for i, (_, row) in enumerate(df.iterrows(), 1):
        print(f"question {i}/{len(df)}")
        response = chain.invoke({"query": row["question"]})
        answers.append(response["result"])
        context = "\n\n---\n\n".join(
            doc.page_content for doc in response["source_documents"]
        )
        contexts.append(context)
 
    df = df.copy()
    df["answer_rag"]      = answers
    df["context_retrieved"] = contexts
    print('generating Q&A done')
    return df


In [12]:
df_aswers = await generate_rag_answers(df_qsts)

question 1/15
question 2/15
question 3/15
question 4/15
question 5/15
question 6/15
question 7/15
question 8/15
question 9/15
question 10/15
question 11/15
question 12/15
question 13/15
question 14/15
question 15/15
generating Q&A done


In [13]:
df_aswers

,question,answer_ref,context,answer_rag,context_retrieved
0,What is the main purpose of the AI surrogate i...,To simulate coastal tidal wave propagation in ...,A Fast AI Surrogate for Coastal Ocean Circulat...,The main purpose of the AI surrogate introduce...,Paper: A Fast AI Surrogate for Coastal Ocean C...
1,What are the limitations of traditional coasta...,"They are time-consuming and energy expensive, ...","However, the process is time-consuming and ene...",I don't know.,Paper: A Fast AI Surrogate for Coastal Ocean C...
2,What technology does the AI surrogate utilize ...,It utilizes a 4D Swin Transformer and incorpor...,This paper introduces an AI surrogate based on...,The AI surrogate utilizes a 4D Swin Transforme...,Paper: A Fast AI Surrogate for Coastal Ocean C...
3,What technology does the AI surrogate use to s...,A 4D Swin Transformer,This paper introduces an AI surrogate based on...,The AI surrogate uses a 4D Swin Transformer to...,Paper: A Fast AI Surrogate for Coastal Ocean C...
4,How much time does the AI surrogate reduce the...,"From 9,908 seconds to 22 seconds, achieving ov...",Our experiments demonstrate that our AI surrog...,The AI surrogate reduces the forecasting time ...,Paper: A Fast AI Surrogate for Coastal Ocean C...
5,What is the primary purpose of simulating ocea...,To develop early warning systems that save liv...,Oceanographers simulate ocean current circulat...,The primary purpose of simulating ocean curren...,Paper: A Fast AI Surrogate for Coastal Ocean C...
6,What is the main focus of the article regardin...,The main focus is on AI (neural network) surro...,The emergence of Graphics Processing Units (GP...,The main focus of the article is the developme...,Paper: A Fast AI Surrogate for Coastal Ocean C...
7,How long does it take to simulate 12 days’ for...,"It takes around 9,908 seconds to simulate 12 d...",Our preliminary results show that with 512 CPU...,"It takes around 9,908 seconds to simulate 12 d...",Paper: A Fast AI Surrogate for Coastal Ocean C...
8,What are the challenges mentioned in applying ...,"Several unique challenges exist, including the...","Unfortunately, these surrogate models cannot b...",The challenges mentioned in applying neural ne...,Paper: A Fast AI Surrogate for Coastal Ocean C...
9,What is the primary focus of the AI surrogate ...,The primary focus of the AI surrogate is to mo...,"For example, vision transformers have been tra...",The primary focus of the AI surrogate introduc...,Paper: A Fast AI Surrogate for Coastal Ocean C...


### 3- Evaluation (now that we have both llm answer and rag answers)

#### 3.1 - LLM as judge

In [ ]:
class CompletenessScore(BaseModel):
    reasoning:   str
    is_complete: bool
    score:       float   # 0 to 1


SYSTEM_COMPLETENESS = """
You are evaluating whether an agent's answer to a scientific question is complete.

A complete answer:
- Covers all key points present in the reference answer
- Does not omit important facts or concepts
- Is based on the provided context

Respond with:
- reasoning: brief explanation
- is_complete: true or false
- score: float between 0 and 1
"""

USER_COMPLETENESS = """
Question: {question}

Reference answer (ground truth): {answer_ref}

Agent answer: {answer_rag}

Context: {context}
"""


async def evaluate_completeness(df: pd.DataFrame):
    """Evaluates the completeness of each rag response against the ground truth"""

    context_col = "context_retrieved" if "context_retrieved" in df.columns else "context"

    sys_msgs  = [SYSTEM_COMPLETENESS] * len(df)
    user_msgs = [
        USER_COMPLETENESS.format(
            question=row["question"],
            answer_ref=row["answer_ref"],
            answer_rag=row["answer_rag"],
            context=row[context_col],
        )
        for _, row in df.iterrows()
    ]

    responses = await run_llm_calls(sys_msgs, user_msgs, response_model=CompletenessScore)

    df = df.copy()
    df["completeness_score"]     = [r.score     if r else None for r in responses]
    df["completeness_reasoning"] = [r.reasoning if r else None for r in responses]
    print('completeness evaluation done')
    return df


In [15]:
df_completeness = await evaluate_completeness(df_aswers)

completeness evaluation done


In [16]:
df_completeness

,question,answer_ref,context,answer_rag,context_retrieved,completeness_score,completeness_reasoning
0,What is the main purpose of the AI surrogate i...,To simulate coastal tidal wave propagation in ...,A Fast AI Surrogate for Coastal Ocean Circulat...,The main purpose of the AI surrogate introduce...,Paper: A Fast AI Surrogate for Coastal Ocean C...,0.6,The agent's answer focuses on the acceleration...
1,What are the limitations of traditional coasta...,"They are time-consuming and energy expensive, ...","However, the process is time-consuming and ene...",I don't know.,Paper: A Fast AI Surrogate for Coastal Ocean C...,0.0,The agent's answer is incomplete as it does no...
2,What technology does the AI surrogate utilize ...,It utilizes a 4D Swin Transformer and incorpor...,This paper introduces an AI surrogate based on...,The AI surrogate utilizes a 4D Swin Transforme...,Paper: A Fast AI Surrogate for Coastal Ocean C...,0.5,The agent's answer mentions the use of a 4D Sw...
3,What technology does the AI surrogate use to s...,A 4D Swin Transformer,This paper introduces an AI surrogate based on...,The AI surrogate uses a 4D Swin Transformer to...,Paper: A Fast AI Surrogate for Coastal Ocean C...,1.0,The agent's answer accurately states that the ...
4,How much time does the AI surrogate reduce the...,"From 9,908 seconds to 22 seconds, achieving ov...",Our experiments demonstrate that our AI surrog...,The AI surrogate reduces the forecasting time ...,Paper: A Fast AI Surrogate for Coastal Ocean C...,0.6,The agent's answer mentions the speedup achiev...
5,What is the primary purpose of simulating ocea...,To develop early warning systems that save liv...,Oceanographers simulate ocean current circulat...,The primary purpose of simulating ocean curren...,Paper: A Fast AI Surrogate for Coastal Ocean C...,1.0,The agent's answer is identical to the referen...
6,What is the main focus of the article regardin...,The main focus is on AI (neural network) surro...,The emergence of Graphics Processing Units (GP...,The main focus of the article is the developme...,Paper: A Fast AI Surrogate for Coastal Ocean C...,0.7,The agent's answer captures the main focus of ...
7,How long does it take to simulate 12 days’ for...,"It takes around 9,908 seconds to simulate 12 d...",Our preliminary results show that with 512 CPU...,"It takes around 9,908 seconds to simulate 12 d...",Paper: A Fast AI Surrogate for Coastal Ocean C...,1.0,The agent's answer is identical to the referen...
8,What are the challenges mentioned in applying ...,"Several unique challenges exist, including the...","Unfortunately, these surrogate models cannot b...",The challenges mentioned in applying neural ne...,Paper: A Fast AI Surrogate for Coastal Ocean C...,1.0,The agent's answer is complete as it includes ...
9,What is the primary focus of the AI surrogate ...,The primary focus of the AI surrogate is to mo...,"For example, vision transformers have been tra...",The primary focus of the AI surrogate introduc...,Paper: A Fast AI Surrogate for Coastal Ocean C...,0.7,The agent's answer focuses on the AI surrogate...


#### 3.2 - RAGAS (faithfulness, relevancy, context_precision)

In [ ]:
def evaluate_ragas_metrics(df: pd.DataFrame) -> pd.DataFrame:
    """
    Evaluate ragas metrics : faithfulness, relevance and faithfulness
    """
    from ragas import evaluate, EvaluationDataset
    from ragas.metrics import faithfulness, answer_relevancy, context_precision
    from ragas.dataset_schema import SingleTurnSample
    from langchain_openai import ChatOpenAI

    langchain_llm = ChatOpenAI(
        model_name=MODEL_NAME,
        openai_api_key=os.environ["OPENAI_API_KEY"],
        temperature=0,
    )

    context_col = "context_retrieved" if "context_retrieved" in df.columns else "context"

    samples = [
        SingleTurnSample(
            user_input=row["question"],
            response=row["answer_rag"],
            retrieved_contexts=[row[context_col]],
            reference=row["answer_ref"],
        )
        for _, row in df.iterrows()
        if row["answer_rag"]
    ]

    dataset = EvaluationDataset(samples=samples)
    results = evaluate(
        dataset,
        metrics=[faithfulness, answer_relevancy, context_precision],
        llm=langchain_llm,
    )

    df = df.copy()
    df["faithfulness"]      = [r.get("faithfulness")      for r in results.scores]
    df["answer_relevancy"]  = [r.get("answer_relevancy")  for r in results.scores]
    df["context_precision"] = [r.get("context_precision") for r in results.scores]
    
    print('ragas evaluation done')
    return df


In [18]:
df_ragas = evaluate_ragas_metrics(df_aswers)

/Users/meriemdahmani/.pyenv/versions/rag_techniques/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/var/folders/80/l8hhvwv56v336dy_j3ctk1vh0000gn/T/ipykernel_54665/2641278820.py:6: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import faithfulness
  from ragas.metrics import faithfulness, answer_relevancy, context_precision
/var/folders/80/l8hhvwv56v336dy_j3ctk1vh0000gn/T/ipykernel_54665/2641278820.py:6: DeprecationWarning: Importing answer_relevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import answer_relevancy
  from ragas.metrics import faithf

ragas evaluation done


In [19]:
df_ragas

,question,answer_ref,context,answer_rag,context_retrieved,faithfulness,answer_relevancy,context_precision
0,What is the main purpose of the AI surrogate i...,To simulate coastal tidal wave propagation in ...,A Fast AI Surrogate for Coastal Ocean Circulat...,The main purpose of the AI surrogate introduce...,Paper: A Fast AI Surrogate for Coastal Ocean C...,1.000000,1.000000,0.0
1,What are the limitations of traditional coasta...,"They are time-consuming and energy expensive, ...","However, the process is time-consuming and ene...",I don't know.,Paper: A Fast AI Surrogate for Coastal Ocean C...,0.000000,0.000000,1.0
2,What technology does the AI surrogate utilize ...,It utilizes a 4D Swin Transformer and incorpor...,This paper introduces an AI surrogate based on...,The AI surrogate utilizes a 4D Swin Transforme...,Paper: A Fast AI Surrogate for Coastal Ocean C...,0.666667,0.963607,1.0
3,What technology does the AI surrogate use to s...,A 4D Swin Transformer,This paper introduces an AI surrogate based on...,The AI surrogate uses a 4D Swin Transformer to...,Paper: A Fast AI Surrogate for Coastal Ocean C...,1.000000,1.000000,1.0
4,How much time does the AI surrogate reduce the...,"From 9,908 seconds to 22 seconds, achieving ov...",Our experiments demonstrate that our AI surrog...,The AI surrogate reduces the forecasting time ...,Paper: A Fast AI Surrogate for Coastal Ocean C...,1.000000,0.994538,1.0
5,What is the primary purpose of simulating ocea...,To develop early warning systems that save liv...,Oceanographers simulate ocean current circulat...,The primary purpose of simulating ocean curren...,Paper: A Fast AI Surrogate for Coastal Ocean C...,1.000000,0.999999,1.0
6,What is the main focus of the article regardin...,The main focus is on AI (neural network) surro...,The emergence of Graphics Processing Units (GP...,The main focus of the article is the developme...,Paper: A Fast AI Surrogate for Coastal Ocean C...,1.000000,0.878637,1.0
7,How long does it take to simulate 12 days’ for...,"It takes around 9,908 seconds to simulate 12 d...",Our preliminary results show that with 512 CPU...,"It takes around 9,908 seconds to simulate 12 d...",Paper: A Fast AI Surrogate for Coastal Ocean C...,1.000000,1.000000,1.0
8,What are the challenges mentioned in applying ...,"Several unique challenges exist, including the...","Unfortunately, these surrogate models cannot b...",The challenges mentioned in applying neural ne...,Paper: A Fast AI Surrogate for Coastal Ocean C...,1.000000,0.994547,1.0
9,What is the primary focus of the AI surrogate ...,The primary focus of the AI surrogate is to mo...,"For example, vision transformers have been tra...",The primary focus of the AI surrogate introduc...,Paper: A Fast AI Surrogate for Coastal Ocean C...,1.000000,1.000000,1.0
